Ячейка 1: Импорты и настройки

In [1]:
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import time
import os

# Входной файл (тот, что на 1 ГБ, 14 млн строк)
input_path = '../data/interim/full_data.parquet'
# Выходной файл (оптимизированный)
output_path = '../data/interim/full_data_optimized_full.parquet'

# Размер чанка (1 миллион строк за раз)
chunk_size = 1_000_000

Ячейка 2: Функция для приведения типов

In [2]:
def optimize_chunk(df):
    schema = {
        'callsign': 'category',
        'typecode': 'category',
        'origin': 'category',
        'destination': 'category',
        'latitude_1': 'float32',
        'longitude_1': 'float32',
        'altitude_1': 'float32',
        'latitude_2': 'float32',
        'longitude_2': 'float32',
        'altitude_2': 'float32',
        'aircraft_uid': 'string'
    }
    # Применяем типы
    df = df.astype(schema)
    # Приводим даты
    df['firstseen'] = pd.to_datetime(df['firstseen'], errors='coerce')
    df['lastseen'] = pd.to_datetime(df['lastseen'], errors='coerce')
    df['day'] = pd.to_datetime(df['day'], errors='coerce')
    return df

Ячейка 3: Прогон по всему файлу с записью на диск

In [3]:
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.compute as pc
import pandas as pd
import time
import os

input_path = '../data/interim/full_data.parquet'
output_path = '../data/interim/full_data_optimized_full.parquet'
chunk_size = 1_000_000

# Безопасное удаление старого файла
if os.path.exists(output_path):
    try:
        os.remove(output_path)
        print("Старый файл успешно удален.")
    except PermissionError:
        print("Файл занят другим процессом. Сохраняем под новым именем...")
        output_path = '../data/interim/full_data_optimized_full_v2.parquet'

print("Начинаем оптимизацию 14 миллионов строк...")
start_time = time.perf_counter()

pf = pq.ParquetFile(input_path)
writer = None

# Читаем самый первый кусочек
first_batch = next(pf.iter_batches(batch_size=100))
df_first = first_batch.to_pandas()

# Применяем типы
schema = {
    'callsign': 'category', 'typecode': 'category',
    'origin': 'category', 'destination': 'category',
    'latitude_1': 'float32', 'longitude_1': 'float32', 'altitude_1': 'float32',
    'latitude_2': 'float32', 'longitude_2': 'float32', 'altitude_2': 'float32',
    'aircraft_uid': 'string'
}
df_first = df_first.astype(schema)
df_first['firstseen'] = pd.to_datetime(df_first['firstseen'], errors='coerce')
df_first['lastseen'] = pd.to_datetime(df_first['lastseen'], errors='coerce')
df_first['day'] = pd.to_datetime(df_first['day'], errors='coerce')

table_first = pa.Table.from_pandas(df_first)
fixed_schema = table_first.schema

# ГЛАВНЫЙ ФИКС: принудительно расширяем индексы категорий до int32, 
# чтобы в них поместились все значения из 14 млн строк
new_fields = []
for field in fixed_schema:
    if pa.types.is_dictionary(field.type):
        # Меняем тип индексов на int32
        new_type = pa.dictionary(pa.int32(), field.type.value_type)
        new_fields.append(pa.field(field.name, new_type))
    else:
        new_fields.append(field)
fixed_schema = pa.schema(new_fields)

# Создаем писатель
writer = pq.ParquetWriter(output_path, fixed_schema)
writer.write_table(table_first.cast(fixed_schema))

print("Схема зафиксирована. Пошла запись...")

for i, batch in enumerate(pf.iter_batches(batch_size=chunk_size)):
    if i == 0 and batch.num_rows <= 100:
        continue
        
    df_chunk = batch.to_pandas()
    df_chunk = df_chunk.astype(schema)
    df_chunk['firstseen'] = pd.to_datetime(df_chunk['firstseen'], errors='coerce')
    df_chunk['lastseen'] = pd.to_datetime(df_chunk['lastseen'], errors='coerce')
    df_chunk['day'] = pd.to_datetime(df_chunk['day'], errors='coerce')
    
    table = pa.Table.from_pandas(df_chunk)
    
    # Приводим к нашей расширенной схеме
    table = table.cast(fixed_schema)
    
    writer.write_table(table)
    print(f"Обработано чанков: {i+1} (строк: {len(df_chunk)})")

writer.close()
end_time = time.perf_counter()
print(f"Готово! Время: {end_time - start_time:.2f} с")

Старый файл успешно удален.
Начинаем оптимизацию 14 миллионов строк...
Схема зафиксирована. Пошла запись...
Обработано чанков: 1 (строк: 1000000)
Обработано чанков: 2 (строк: 1000000)
Обработано чанков: 3 (строк: 1000000)
Обработано чанков: 4 (строк: 1000000)
Обработано чанков: 5 (строк: 1000000)
Обработано чанков: 6 (строк: 1000000)
Обработано чанков: 7 (строк: 1000000)
Обработано чанков: 8 (строк: 1000000)
Обработано чанков: 9 (строк: 1000000)
Обработано чанков: 10 (строк: 1000000)
Обработано чанков: 11 (строк: 1000000)
Обработано чанков: 12 (строк: 1000000)
Обработано чанков: 13 (строк: 1000000)
Обработано чанков: 14 (строк: 1000000)
Обработано чанков: 15 (строк: 9751)
Готово! Время: 29.33 с


Ячейка 4: Проверка результата

In [4]:
# Смотрим размер нового файла
old_size = os.path.getsize(input_path) / (1024 * 1024)
new_size = os.path.getsize(output_path) / (1024 * 1024)

print(f"Размер ДО (14 млн строк): {old_size:.2f} МБ")
print(f"Размер ПОСЛЕ (14 млн строк): {new_size:.2f} МБ")
print(f"Экономия: {100 - (new_size / old_size * 100):.2f}%")

# Читаем первые 5 строк для проверки
df_check = pd.read_parquet(output_path).head()
print("\nТипы данных в новом файле:")
print(df_check.dtypes)

Размер ДО (14 млн строк): 1086.44 МБ
Размер ПОСЛЕ (14 млн строк): 944.80 МБ
Экономия: 13.04%

Типы данных в новом файле:
callsign                   category
number                          str
aircraft_uid                    str
typecode                   category
origin                     category
destination                category
firstseen       datetime64[us, UTC]
lastseen        datetime64[us, UTC]
day             datetime64[us, UTC]
latitude_1                  float32
longitude_1                 float32
altitude_1                  float32
latitude_2                  float32
longitude_2                 float32
altitude_2                  float32
dtype: object


Таблица 7 - Результаты оптимизации схемы данных

| Показатель | До оптимизации | После | Изменение |
| :--- | :--- | :--- | :--- |
| Объём файла на диске, МБ | 1086.44 | 944.80 | -141.64 МБ (-13.04%) |
| Объём в памяти, МБ | 468.96 | 212.33 | -256.63 МБ (-54.72%) |
| Время чтения, с | 2.91 | 2.50 | -0.41 с |
| Время группировки, с | 0.0815 | 0.0675 | -0.0140 с (в 1.2 раза быстрее) |
| Число столбцов типа category | 0 | 4 | +4 |